# Reference unfragmented CCSD(T) geometry optimization

This notebook performs a **reference geometry optimization of propylene at the unfragmented (full-molecule) CCSD(T) level** with analytic nuclear gradients. It serves as the benchmark against which the results of the EWF simulations in this repository are compared: because no fragmentation or embedding is involved, the optimized geometry obtained here is free of the approximations introduced by the EWF density assembly, and any deviation of an EWF-optimized structure from this one measures the error of the embedding treatment itself.

The optimized geometry produced at the end of this notebook is the `propylene_ccsd_t.txt` reference used by [`../Geom_Comparison_Tool/`](../Geom_Comparison_Tool/) to compute RMSD and per-atom deviations of the `rdm_t` and `rdm_t_lambda` EWF geometries.

**Settings are chosen to match the EWF runs** (`../source/config.yaml`): the same input geometry (`propylene.txt`), the same STO-3G basis, charge 0, singlet, no symmetry — so the comparison isolates the effect of fragmentation, not of the basis or reference.

**Workflow:**
1. Read the starting geometry and build the PySCF molecule.
2. Set up the RHF mean-field reference.
3. Define a CCSD(T) energy + analytic gradient scanner.
4. Optimize the geometry with geomeTRIC.
5. Print the converged reference geometry.

## 1. Input geometry and molecule setup

Read the starting propylene structure from `propylene.txt` (plain `Element x y z` format, Angstrom) and build the `gto.Mole` object.

In [1]:
from pyscf import gto

def read_geometry(fname):
    f = list(open(fname,'r').readlines())
    f = [fi.split() for fi in f]
    f = [[fi[0],(float(fi[1]),float(fi[2]),float(fi[3]))] for fi in f]
    return f

# ---------- Your structure

geo = read_geometry('propylene.txt')
mol = gto.Mole()
mol.build(     
    atom       = geo,
    basis      = '6-31G*',
    verbose    = 0,
    charge     = 0,
    spin       = 0,
    symmetry   = False)

## 2. Mean-field reference (RHF)

Restricted Hartree–Fock provides the reference determinant for the coupled-cluster treatment. The actual SCF at each optimization step is re-run inside the scanner below; this cell just establishes the mean-field object for the initial geometry.

In [2]:
from pyscf import scf
mf = scf.RHF(mol)

## 3. CCSD(T) energy + analytic gradient scanner

`ccsd_t_scanner` evaluates, at each candidate geometry supplied by the optimizer:

1. **RHF** — re-converge the mean field.
2. **CCSD** — solve the coupled-cluster ground state.
3. **(T)** — add the perturbative-triples energy correction (`ccsd_t()`).
4. **Λ equations** — solve the CCSD(T) lambda equations (`ccsd_t_lambda`), which provide the relaxed density required for analytic gradients.
5. **Analytic nuclear gradient** — contract the relaxed densities with the integral derivatives (`pyscf.grad.ccsd_t`).

Note the contrast with the EWF workflow: here the Λ equations are solved exactly for the full molecule, so the gradient is the exact derivative of the CCSD(T) energy — there is no fragmentation, no density assembly, and therefore no missing density-response term (see the discussion in the repository [`README.md`](../README.md)).

`as_pyscf_method` wraps the scanner so PySCF's geometry-optimizer interface can drive it.

In [3]:
from pyscf import cc
from pyscf.cc import ccsd_t_lambda_slow as ccsd_t_lambda
from pyscf.grad import ccsd_t as ccsd_t_grad
from pyscf.geomopt.addons import as_pyscf_method

#Build the energy and gradient scanner function
def ccsd_t_scanner(mol_input):
    """
    Computes both the total CCSD(T) energy and nuclear gradients
    for a given molecular configuration.
    """
    # Mean-Field Hartree-Fock
    mf = scf.RHF(mol_input).run()
    
    # Ground-state CCSD
    mycc = cc.CCSD(mf).run()
    
    # Calculate energy correction for perturbative triples (T)
    et = mycc.ccsd_t()
    e_tot = mycc.e_tot + et
    
    # Resolve the CCSD(T) Lambda equations required for gradients
    eris = mycc.ao2mo()
    conv, l1, l2 = ccsd_t_lambda.kernel(mycc, eris, mycc.t1, mycc.t2)
    
    # Compute the analytical nuclear gradients
    de = ccsd_t_grad.Gradients(mycc).kernel(mycc.t1, mycc.t2, l1, l2, eris=eris)
    
    return e_tot, de

#Adapt the scanner to make it compatible with PySCF geometry optimizers
pyscf_method_wrapper = as_pyscf_method(mol, ccsd_t_scanner)

## 4. Geometry optimization (geomeTRIC)

Run the optimization with PySCF's geomeTRIC interface. The convergence criteria listed below are geomeTRIC's defaults (Gaussian-style thresholds), written out explicitly for transparency. Because the CCSD(T) gradient is exact, the optimizer can be held to these **tight criteria** — unlike the EWF optimizations, which currently require looser thresholds to accommodate the residual gradient floor of the embedding.

In [4]:
# geometric
from pyscf.geomopt.geometric_solver import optimize

conv_params = { # These are the default settings
    'convergence_energy': 1e-6,  # Eh
    'convergence_grms': 3e-4,    # Eh/Bohr
    'convergence_gmax': 4.5e-4,  # Eh/Bohr
    'convergence_drms': 1.2e-3,  # Angstrom
    'convergence_dmax': 1.8e-3,  # Angstrom
}
mol_eq = optimize(pyscf_method_wrapper, **conv_params)

geometric-optimize called with the following command line:
/opt/homebrew/Caskroom/miniforge/base/envs/classical/lib/python3.12/site-packages/ipykernel_launcher.py --f=/Users/kaliakd/Library/Jupyter/runtime/kernel-v35968078dc0a99e436d785eb5c21294de5aaa85aa.json

                                        ())))))))))))))))/                     
                                    ())))))))))))))))))))))))),                
                                *)))))))))))))))))))))))))))))))))             
                        #,    ()))))))))/                .)))))))))),          
                      #%%%%,  ())))))                        .))))))))*        
                      *%%%%%%,  ))              ..              ,))))))).      
                        *%%%%%%,         ***************/.        .)))))))     
                #%%/      (%%%%%%,    /*********************.       )))))))    
              .%%%%%%#      *%%%%%%,  *******/,     **********,      .))))))   
                .%

cycle = 1  norm(lambda1,lambda2) = 0.0380854
cycle = 2  norm(lambda1,lambda2) = 0.00749738
cycle = 3  norm(lambda1,lambda2) = 0.00402924
cycle = 4  norm(lambda1,lambda2) = 0.00192332
cycle = 5  norm(lambda1,lambda2) = 0.000303242
cycle = 6  norm(lambda1,lambda2) = 0.000109647
cycle = 7  norm(lambda1,lambda2) = 2.62309e-05
cycle = 8  norm(lambda1,lambda2) = 7.41535e-06
cycle = 9  norm(lambda1,lambda2) = 3.07396e-06
cycle = 10  norm(lambda1,lambda2) = 8.49392e-07
cycle = 11  norm(lambda1,lambda2) = 2.37818e-07
cycle = 12  norm(lambda1,lambda2) = 8.94543e-08
cycle = 13  norm(lambda1,lambda2) = 2.99973e-08
cycle = 14  norm(lambda1,lambda2) = 1.89254e-08
cycle = 15  norm(lambda1,lambda2) = 1.35878e-08
cycle = 16  norm(lambda1,lambda2) = 1.18283e-08
cycle = 17  norm(lambda1,lambda2) = 1.04575e-08
cycle = 18  norm(lambda1,lambda2) = 1.36701e-08
cycle = 19  norm(lambda1,lambda2) = 8.57064e-09


Step    0 : Gradient = 4.401e-02/5.091e-02 (rms/max) Energy = -117.4694977568
Hessian Eigenvalues: 2.30000e-02 2.30000e-02 2.54027e-02 ... 2.58307e-01 2.58467e-01 3.99459e-01


cycle = 1  norm(lambda1,lambda2) = 0.0305066
cycle = 2  norm(lambda1,lambda2) = 0.00593841
cycle = 3  norm(lambda1,lambda2) = 0.00296041
cycle = 4  norm(lambda1,lambda2) = 0.00128609
cycle = 5  norm(lambda1,lambda2) = 0.000195036
cycle = 6  norm(lambda1,lambda2) = 6.53766e-05
cycle = 7  norm(lambda1,lambda2) = 1.62968e-05
cycle = 8  norm(lambda1,lambda2) = 4.406e-06
cycle = 9  norm(lambda1,lambda2) = 1.74313e-06
cycle = 10  norm(lambda1,lambda2) = 4.69881e-07
cycle = 11  norm(lambda1,lambda2) = 1.27133e-07
cycle = 12  norm(lambda1,lambda2) = 4.9999e-08
cycle = 13  norm(lambda1,lambda2) = 2.88159e-08
cycle = 14  norm(lambda1,lambda2) = 2.17014e-08
cycle = 15  norm(lambda1,lambda2) = 1.80738e-08
cycle = 16  norm(lambda1,lambda2) = 1.59863e-08
cycle = 17  norm(lambda1,lambda2) = 8.81937e-09


Step    1 : Displace = 1.008e-01/1.367e-01 (rms/max) Trust = 1.000e-01 (=) Grad = 1.269e-02/1.571e-02 (rms/max) E (change) = -117.5086675235 (-3.917e-02) Quality = 1.007
Hessian Eigenvalues: 2.30000e-02 2.30000e-02 2.54027e-02 ... 2.57995e-01 2.58727e-01 4.38339e-01


...

Step    5 : Displace = 3.122e-03/4.853e-03 (rms/max) Trust = 3.549e-02 (+) Grad = 4.567e-04/7.190e-04 (rms/max) E (change) = -117.5110454046 (-1.206e-05) Quality = 0.612
Hessian Eigenvalues: 2.30000e-02 2.30000e-02 2.54027e-02 ... 3.52703e-01 4.22609e-01 5.79505e-01


cycle = 1  norm(lambda1,lambda2) = 0.0292686
cycle = 2  norm(lambda1,lambda2) = 0.00578486
cycle = 3  norm(lambda1,lambda2) = 0.00288201
cycle = 4  norm(lambda1,lambda2) = 0.00123291
cycle = 5  norm(lambda1,lambda2) = 0.000181934
cycle = 6  norm(lambda1,lambda2) = 6.04039e-05
cycle = 7  norm(lambda1,lambda2) = 1.52035e-05
cycle = 8  norm(lambda1,lambda2) = 4.07544e-06
cycle = 9  norm(lambda1,lambda2) = 1.63489e-06
cycle = 10  norm(lambda1,lambda2) = 4.41977e-07
cycle = 11  norm(lambda1,lambda2) = 1.20076e-07
cycle = 12  norm(lambda1,lambda2) = 4.69616e-08
cycle = 13  norm(lambda1,lambda2) = 2.69684e-08
cycle = 14  norm(lambda1,lambda2) = 2.04112e-08
cycle = 15  norm(lambda1,lambda2) = 1.68315e-08
cycle = 16  norm(lambda1,lambda2) = 1.49755e-08
cycle = 17  norm(lambda1,lambda2) = 1.86895e-08
cycle = 18  norm(lambda1,lambda2) = 1.13922e-08
cycle = 19  norm(lambda1,lambda2) = 8.96609e-09


Step    6 : Displace = 1.418e-03/2.448e-03 (rms/max) Trust = 3.549e-02 (=) Grad = 3.774e-05/8.474e-05 (rms/max) E (change) = -117.5110476371 (-2.233e-06) Quality = 0.972
Hessian Eigenvalues: 2.30000e-02 2.30000e-02 2.54027e-02 ... 3.48823e-01 4.29324e-01 5.78033e-01


cycle = 1  norm(lambda1,lambda2) = 0.0292716
cycle = 2  norm(lambda1,lambda2) = 0.0057854
cycle = 3  norm(lambda1,lambda2) = 0.00288242
cycle = 4  norm(lambda1,lambda2) = 0.00123316
cycle = 5  norm(lambda1,lambda2) = 0.000181969
cycle = 6  norm(lambda1,lambda2) = 6.04196e-05
cycle = 7  norm(lambda1,lambda2) = 1.52068e-05
cycle = 8  norm(lambda1,lambda2) = 4.0764e-06
cycle = 9  norm(lambda1,lambda2) = 1.63543e-06
cycle = 10  norm(lambda1,lambda2) = 4.42126e-07
cycle = 11  norm(lambda1,lambda2) = 1.20113e-07
cycle = 12  norm(lambda1,lambda2) = 4.69697e-08
cycle = 13  norm(lambda1,lambda2) = 2.7021e-08
cycle = 14  norm(lambda1,lambda2) = 2.04182e-08
cycle = 15  norm(lambda1,lambda2) = 1.68375e-08
cycle = 16  norm(lambda1,lambda2) = 1.49791e-08
cycle = 17  norm(lambda1,lambda2) = 1.86986e-08
cycle = 18  norm(lambda1,lambda2) = 1.13987e-08
cycle = 19  norm(lambda1,lambda2) = 8.97224e-09


Step    7 : Displace = 4.434e-04/8.618e-04 (rms/max) Trust = 5.019e-02 (+) Grad = 3.433e-05/5.324e-05 (rms/max) E (change) = -117.5110476309 (+6.149e-09) Quality = -0.097
Hessian Eigenvalues: 2.30000e-02 2.30000e-02 2.54027e-02 ... 3.48823e-01 4.29324e-01 5.78033e-01
Converged! =D

    #==========================================================================#
    #| If this code has benefited your research, please support us by citing: |#
    #|                                                                        |#
    #| Wang, L.-P.; Song, C.C. (2016) "Geometry optimization made simple with |#
    #| translation and rotation coordinates", J. Chem, Phys. 144, 214108.     |#
    #| http://dx.doi.org/10.1063/1.4952956                                    |#
    #==========================================================================#
    Time elapsed since start of run_optimizer: 273.679 seconds


## 5. Optimized reference geometry

The converged CCSD(T) structure (Angstrom). This is the **reference geometry** for benchmarking the EWF results: it is stored as `propylene_ccsd_t.txt` in [`../Geom_Comparison_Tool/`](../Geom_Comparison_Tool/), where the geometries optimized with the `rdm_t` and `rdm_t_lambda` EWF assemblies are compared against it via Kabsch-aligned RMSD and per-atom deviations:

```bash
cd ../Geom_Comparison_Tool
python geom_compare.py propylene_ccsd_t.txt propylene_rdm_t.txt propylene_rdm_t_lambda.txt
```

In [5]:
print(mol_eq.tostring())

C           1.21406767       -0.18970967       -0.00000000
C          -0.14849040        0.44967050       -0.00000000
C          -1.30887402       -0.22770209       -0.00000000
H           1.79291987        0.11318281       -0.88453502
H           1.13470323       -1.28453098       -0.00000000
H           1.79291987        0.11318281        0.88453502
H          -0.17132830        1.54306536       -0.00000000
H          -2.27086597        0.28307607       -0.00000000
H          -1.33431929       -1.31837414       -0.00000000
